# Step 1: Environment Setup & Data Preprocessing

This notebook will guide you through verifying your local or Colab Python environment, downloading the Chest X-ray datasets for Tuberculosis, and loading them into memory for processing.

### 1. Verification of Libraries and CUDA (GPU)
Run the following code block to verify that PyTorch is installed correctly and can see your NVIDIA GPU (if available).

In [ ]:
import sys
import torch
import torchvision
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"OpenCV Version: {cv2.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"CUDA Available (GPU Support): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

### 2. Dataset Information

For Tuberculosis & Pneumonia detection, we use a combined dataset containing:
1. **NORMAL** CXR scans (healthy lung fields)
2. **TUBERCULOSIS** CXR scans (active tuberculosis lesions)
3. **PNEUMONIA** CXR scans (bacterial or viral pneumonia opacities)

#### Dataset Location:
- Downloaded dataset splits and images are located at: `E:\Download\TB Data\data`
- Class directories: `NORMAL`, `TUBERCULOSIS`, `PNEUMONIA` (under `train`, `val`, and `test` splits).

### 3. Load Datasets & Build Metadata CSV
Run this block once you have populated the `data/raw/` directories to generate a metadata CSV containing image paths and their corresponding labels (0 for Normal, 1 for Tuberculosis).

In [ ]:
import os
from pathlib import Path

# Define dataset path
DATA_DIR = Path("E:/Download/TB Data/data")

# Check if directories exist
print(f"Dataset root directory exists: {DATA_DIR.exists()}")
for split in ['train', 'val', 'test']:
    for label in ['NORMAL', 'TUBERCULOSIS', 'PNEUMONIA']:
        p = DATA_DIR / split / label
        print(f"  - Folder '{split}/{label}' exists: {p.exists()} ({len(list(p.glob('*'))) if p.exists() else 0} files)")

In [ ]:
import pandas as pd
import os
from pathlib import Path

def build_metadata_csv():
    downloaded_csv = Path(r"E:\Download\TB Data\data\dataset_splits_with_metadata.csv")
    dest_dir = Path("../data") if Path("../data").exists() else Path("data")
    dest_csv = dest_dir / "metadata.csv"
    
    if not downloaded_csv.exists():
        print(f"Please verify dataset splits CSV exists at {downloaded_csv}!")
        return
        
    print(f"[+] Reading downloaded metadata from: {downloaded_csv}")
    df = pd.read_csv(downloaded_csv)
    
    # Filter out UNKNOWN class
    df = df[df['label'] != 'UNKNOWN'].copy()
    
    # Construct relative path
    df['image_path'] = df['split'] + '/' + df['label'] + '/' + df['name']
    
    # Map text labels to numeric labels
    label_map = {
        'NORMAL': 0,
        'TUBERCULOSIS': 1,
        'PNEUMONIA': 2
    }
    df['class_name'] = df['label']
    df['label'] = df['label'].map(label_map)
    
    # Reorder columns
    df = df[['image_path', 'label', 'class_name', 'split', 'source']]
    
    os.makedirs(dest_dir, exist_ok=True)
    df.to_csv(dest_csv, index=False)
    print(f"Successfully generated metadata.csv with {len(df)} images at {dest_csv}")
    print(df["class_name"].value_counts())

build_metadata_csv()

### 4. Plotting Sample Images
Once the metadata file is built, run this block to visualize some sample Chest X-rays from both classes.

In [ ]:
metadata_file = Path("../data/metadata.csv") if Path("../data/metadata.csv").exists() else Path("data/metadata.csv")
DATA_ROOT = Path("E:/Download/TB Data/data")

if metadata_file.exists():
    df = pd.read_csv(metadata_file)
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    
    # Get 4 samples for each class
    normal_samples = df[df["label"] == 0].head(4)
    tb_samples = df[df["label"] == 1].head(4)
    pneumonia_samples = df[df["label"] == 2].head(4)
    
    for i, (_, row) in enumerate(normal_samples.iterrows()):
        img = cv2.imread(str(DATA_ROOT / row['image_path']))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax = axes[0, i]
        ax.imshow(img)
        ax.set_title("Normal")
        ax.axis('off')
        
    for i, (_, row) in enumerate(tb_samples.iterrows()):
        img = cv2.imread(str(DATA_ROOT / row['image_path']))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax = axes[1, i]
        ax.imshow(img)
        ax.set_title("Tuberculosis")
        ax.axis('off')
        
    for i, (_, row) in enumerate(pneumonia_samples.iterrows()):
        img = cv2.imread(str(DATA_ROOT / row['image_path']))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax = axes[2, i]
        ax.imshow(img)
        ax.set_title("Pneumonia")
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("metadata.csv not found. Please run Step 3 first.")